In [1]:
%load_ext rpy2.ipython

In [2]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import func
from sqlalchemy.orm import Query

import src
from src.data.models import Channel
from src.data.models import Comment
from src.data.models import Sentence
from src.data.models import Video

In [3]:
pd.options.display.float_format = "{:.1f}".format

colormap = pd.DataFrame(src.colormap.items(), columns=["channel", "color"])

engine = create_engine(src.PS_ENGINE)

# per Channel

In [4]:
query_all = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.format == "videos",
    )
    .with_entities(
        Channel.channel,
        Channel.channel_follower_count.label("followers"),
        # func.count(Video.id.distinct()).label("videos"),
        # func.count(Sentence.id).label("sentences"),
        # func.min(Video.datetime_upload).label("first_video"),
    )
    .order_by(Channel.channel_follower_count.desc())
)

query_after = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.is_valid == True,
        Sentence.is_valid == True,
    )
    .with_entities(
        Channel.channel,
        func.count(Video.id.distinct()).label("videos"),
        func.count(Sentence.id).label("sentences"),
    )
)


with engine.connect() as conn:
    df_all = pd.read_sql(query_all.statement, conn)
    df_after = pd.read_sql(query_after.statement, conn)

df = pd.merge(df_all, df_after, on="channel")
df.channel = df.channel.replace(
    {"BÜNDNIS 90/DIE GRÜNEN": "Grüne", "AfD-Fraktion Bundestag": "AfD BT"},
)
df

,channel,followers,videos,sentences
0,AfD BT,388000,5228,300196
1,AfD TV,250000,1461,145642
2,DIE LINKE,29000,436,46840
3,Grüne,26100,460,47016
4,SPD,24200,484,67174
5,FDP,23300,487,36930
6,CDU,21900,630,54288
7,CSU,5170,145,10463


In [5]:
df_channel = df.copy()

# per Video

In [6]:
comments_query = (
    Query(Comment.id, func.count(Comment.id))
    .filter(Comment.video_id == Video.id, Comment.is_valid == True)
    .with_entities(func.count(Comment.id))
    .scalar_subquery()
)
query = (
    Query(Video)
    .join(Channel)
    .filter(Video.is_valid == True)
    .with_entities(
        Channel.channel,
        Video.datetime_upload,
        Video.like_count.label("likes"),
        Video.view_count.label("views"),
        Video.duration,
        comments_query.label("comments"),
    )
)

with engine.connect() as conn:
    df = pd.read_sql(query.statement, conn)
df.channel = df.channel.replace(
    {"BÜNDNIS 90/DIE GRÜNEN": "Grüne", "AfD-Fraktion Bundestag": "AfD BT", "DIE LINKE": "Linke"},
)

In [7]:
df_videos = df.groupby("channel").mean(numeric_only=True).stack().reset_index()

pivot_videos = pd.pivot(df_videos, index="channel", columns="level_1", values=0)
pivot_videos.columns = [f"mean_{col.lstrip('@')}" for col in pivot_videos.columns]
pivot_videos = pivot_videos.reset_index()

In [8]:
pivot_videos

,channel,mean_comments,mean_duration,mean_likes,mean_views
0,AfD BT,398.9,435.6,3876.7,45910.4
1,AfD TV,398.6,655.9,3624.6,43293.2
2,CDU,41.1,615.2,64.3,9609.8
3,CSU,7.5,442.2,35.6,22332.8
4,FDP,0.7,620.3,0.5,5874.0
5,Grüne,0.2,890.8,79.5,4528.0
6,Linke,45.1,830.9,255.5,10406.1
7,SPD,25.1,1082.1,101.8,5031.8


In [9]:
summary_table = pd.merge(df_channel, pivot_videos, on="channel").T
summary_table.columns = [col.lstrip("@") for col in summary_table.iloc[0,:]]
summary_table = summary_table.iloc[1:,:]
summary_table = summary_table

In [10]:
summary_table

,AfD BT,AfD TV,Grüne,SPD,FDP,CDU,CSU
followers,388000,250000,26100,24200,23300,21900,5170
videos,5228,1461,460,484,487,630,145
sentences,300196,145642,47016,67174,36930,54288,10463
mean_comments,398.9,398.6,0.2,25.1,0.7,41.1,7.5
mean_duration,435.6,655.9,890.8,1082.1,620.3,615.2,442.2
mean_likes,3876.7,3624.6,79.5,101.8,0.5,64.3,35.6
mean_views,45910.4,43293.2,4528.0,5031.8,5874.0,9609.8,22332.8


In [11]:
summary_table.to_latex(
    src.PATH / "overleaf/tables/summary.tex",
    float_format= "{:.1f}".format,
    escape=True,
    column_format="lrrrrrrr",
)

In [12]:
%%R -i df -i colormap -w 1000 -h 600

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)

options(scipen = 999)

cmap <- setNames(colormap$color, colormap$channel)

ggplot(df, aes(x=channel, y=views, fill=channel)) +
   geom_violin(draw_quantiles=c(0.25, 0.5, 0.75), alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1)
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")


ggsave(here("overleaf/img/view_count_violin.pdf"))

Saving 13.9 x 8.33 in image


here() starts at /Users/lukas/git/ytpop
